# NeuroGolf 2026 - Optimal Blending Ensemble

This notebook iterates through all attached datasets (which can include the outputs of your 4000+ scoring notebooks), extracts all candidate ONNX models for each task, validates them against the task JSON, and selects the optimal (smallest valid) model for submission.

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import json, os, zipfile, shutil
from pathlib import Path
import numpy as np
import onnxruntime as ort
import onnx
import onnx.numpy_helper as onh

ort.set_default_logger_severity(3)

# Locate the competition data directory (contains the task JSONs)
TASK_DIR = None
for p in Path('/kaggle/input').rglob('task001.json'):
    TASK_DIR = p.parent
    break

print('TASK_DIR:', TASK_DIR)

OUT = Path('/kaggle/working/submission')
OUT.mkdir(exist_ok=True)
TEMP_CANDIDATES = Path('/kaggle/working/candidates')
TEMP_CANDIDATES.mkdir(exist_ok=True)

candidates = {}
cand_idx = 0

# Collect standalone ONNX files
for p in Path('/kaggle/input').rglob('task*.onnx'):
    try:
        tn = int(p.stem.replace('task',''))
        cand_path = TEMP_CANDIDATES / f'cand_{cand_idx}.onnx'
        shutil.copy(p, cand_path)
        candidates.setdefault(tn, []).append(cand_path)
        cand_idx += 1
    except:
        pass

# Collect ONNX files from ZIP archives (like submission.zip from other notebooks)
for z in Path('/kaggle/input').rglob('*.zip'):
    try:
        with zipfile.ZipFile(z) as zf:
            for name in zf.namelist():
                bn = Path(name).name
                if bn.startswith('task') and bn.endswith('.onnx'):
                    try:
                        tn = int(bn.replace('task','').replace('.onnx',''))
                        cand_path = TEMP_CANDIDATES / f'cand_{cand_idx}.onnx'
                        cand_path.write_bytes(zf.read(name))
                        candidates.setdefault(tn, []).append(cand_path)
                        cand_idx += 1
                    except:
                        pass
    except:
        pass

print(f'Total tasks with candidates: {len(candidates)}')


def g2t(grid):
    t = np.zeros((1, 10, 30, 30), np.float32)
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            if 0 <= v < 10 and r < 30 and c < 30:
                t[0, v, r, c] = 1
    return t

def verify(path, tn):
    try:
        sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    except Exception:
        return False
        
    if TASK_DIR is None:
        return True
        
    try:
        d = json.load(open(TASK_DIR / f'task{tn:03d}.json'))
    except Exception:
        return False
        
    for p in d.get('train', []) + d.get('test', []) + d.get('arc-gen', []):
        try:
            out = sess.run(None, {'input': g2t(p['input'])})[0]
            pred = (out > 0).astype(np.float32)
            if not np.array_equal(pred, g2t(p['output'])):
                return False
        except Exception:
            return False
    return True

def get_params(path):
    try:
        model = onnx.load(str(path))
        return sum(onh.to_array(i).size for i in model.graph.initializer)
    except Exception:
        return float('inf')


In [ ]:
valid_count = 0

for tn, cands in candidates.items():
    best_cand = None
    best_params = float('inf')
    
    for cand in cands:
        if verify(cand, tn):
            params = get_params(cand)
            # We prefer models with fewer parameters to stay under Kaggle's 1.44 MB limit
            if params < best_params:
                best_params = params
                best_cand = cand
                
    if best_cand is not None:
        valid_count += 1
        dst = OUT / f'task{tn:03d}.onnx'
        shutil.copy(best_cand, dst)

print(f'Valid tasks successfully blended and kept: {valid_count}')


In [ ]:
# Package the final optimal submission
zp = Path('/kaggle/working/submission.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.glob('*.onnx')):
        zf.write(f, f.name)

print(f'submission.zip created: {zp.stat().st_size/1024:.1f} KB, {len(list(OUT.glob("*.onnx")))} files')

# Cleanup temporary candidates
shutil.rmtree(TEMP_CANDIDATES, ignore_errors=True)
